# SMF 99 · Consistency check after a catalog update

One-off check made when the master catalog was rebuilt: are the most massive members of each cluster in the previous catalog still present in the new one? Kept for transparency; it depends on the previous catalog version, which is not part of the repository.

In [1]:
# ============================================================================
# Setup
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from astropy.cosmology import LambdaCDM

# Cosmology used throughout this project
cosmo = LambdaCDM(H0=70, Om0=0.3, Ode0=0.7)

# Matplotlib style shared by the notebooks of this repository
plt.rcParams.update({
    "font.family": 'STIXGeneral',
    'text.usetex': False,
    "mathtext.fontset": 'cm',
    "axes.labelweight": "normal",
    'font.size': 25,
    'font.weight': 'normal',
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.major.size': 10,
    'xtick.minor.size': 6,
    'ytick.major.size': 10,
    'ytick.minor.size': 6,
    'xtick.major.width': 1.6,
    'xtick.minor.width': 1.6,
    'ytick.major.width': 1.6,
    'ytick.minor.width': 1.6,
    'lines.linewidth': 2,
    'axes.linewidth': 4,
    'axes.labelpad': 4,
    'xtick.major.pad': 7,
    'image.origin': 'lower'
})

# Show every column when a DataFrame is displayed
pd.set_option('display.max_columns', None)

# The nine MACH clusters
MACHLIST = ['A2245', 'A1767', 'A2244', 'A1831', 'A2034', 'A7', 'A2255', 'A2029', 'A2065']

# Repository root, located relative to this notebook; DATA/ and FIGURE/ hang off it
MAINPATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
FIGUREPATH = f"{MAINPATH}/FIGURE/IMAGES"

%config InlineBackend.figure_format = 'retina'

In [2]:
# Load MACH master catalog
mach_path = f"{MAINPATH}/DATA/MACH_MASS/MACH_DR18_3R200_mastercat.csv"
mach_data = pd.read_csv(mach_path)

# Correct cmodel magnitudes for extinction
for band in ("g", "r"):
    mach_data[f"cmodelMag_{band}_0"] = (
        mach_data[f"cmodelMag_{band}"] - mach_data[f"extinction_{band}"]
    )

# Load HeCS VAC catalog
hecs_vac_path = f"{MAINPATH}/DATA/MACH_PHOT/AllHeCS_VAC_updated.csv"
hecs_vac_data = pd.read_csv(hecs_vac_path)

# Galaxies within 3 R200
df = mach_data.loc[mach_data["Rcl/R200"] <= 3.0].copy()

# Construct Mock_z: use Z_TOT_Z, but if Z_TOT_Z == -99, use cluster mean z from hecs_vac_data (by CLID)
df["Mock_z"] = df["Z_TOT_Z"]

# Find rows with Z_TOT_Z == -99
mask_noz = (df["Z_TOT_Z"] == -99)

# For those rows, replace with matching cluster z from hecs_vac_data
if "CLID" in df.columns and "CLID" in hecs_vac_data.columns:
    # Construct cluster z dictionary from hecs_vac_data
    clid_to_z = hecs_vac_data.set_index("CLID")["Z"].to_dict()
    # Fill Mock_z for those without z
    df.loc[mask_noz, "Mock_z"] = df.loc[mask_noz, "CLID"].map(clid_to_z)

# Compute absolute r-band magnitude using Mock_z
lum_dist_pc = cosmo.luminosity_distance(df["Mock_z"]).to("pc").value
df["AbsMag_r"] = df["cmodelMag_r_0"] - 5 * np.log10(lum_dist_pc / 10)

/var/folders/8j/r5cxq2xj3bxdkw71t64p365m0000gn/T/ipykernel_70454/2657588277.py:34: RuntimeWarning: divide by zero encountered in log10
  df["AbsMag_r"] = df["cmodelMag_r_0"] - 5 * np.log10(lum_dist_pc / 10)
/var/folders/8j/r5cxq2xj3bxdkw71t64p365m0000gn/T/ipykernel_70454/2657588277.py:34: RuntimeWarning: invalid value encountered in log10
  df["AbsMag_r"] = df["cmodelMag_r_0"] - 5 * np.log10(lum_dist_pc / 10)


In [3]:
# Raw SDSS photometric catalog (not used below; loaded for interactive checks)
rawcat = pd.read_csv(f'{MAINPATH}/DATA/MACH_PHOT/SDSS_RAWCAT_DR18/MACH_SDSS_DR18_within3R200_rawcat.csv')

In [7]:
# Previous version of the master catalog (not included)
df_old = pd.read_csv(f'{MAINPATH}/DATA/MACH_MASS/MACH_DR18_within3R200_mastercat_old0220.csv')

# Ten most massive members per cluster in the previous catalog
top3_by_clid_old = (
    df_old[df_old['MEMBER'] == 'Y']
    .sort_values(['CLID', 'M_MASS_CIGALE_PARK'], ascending=[True, False])
    .groupby('CLID')
    .head(10)
    .loc[:, ['objid', 'CLID', 'MEMBER', 'M_MASS_CIGALE_PARK', 'Rcl/R200']]
)

# Those that are missing from the new catalog (by objid)
top3_missing_in_new = top3_by_clid_old[~top3_by_clid_old['objid'].isin(df['objid'])]
top3_missing_in_new

/var/folders/8j/r5cxq2xj3bxdkw71t64p365m0000gn/T/ipykernel_70454/2819250961.py:1: DtypeWarning: Columns (50,61,67,70,71,75,76,85,86) have mixed types. Specify dtype option on import or set low_memory=False.
  df_old = pd.read_csv('/Users/jonginpark/sohnix/Research_Archive/2026/Park2026b-MACH-SMF/DATA/MACH_MASS/MACH_DR18_within3R200_mastercat_old0220.csv')


,objid,CLID,MEMBER,M_MASS_CIGALE_PARK,Rcl/R200
212573,1237665442595209429,A1831,Y,11.525223,0.571687
212742,1237662306194423863,A2034,Y,11.659169,0.198880
212729,1237662194004656247,A2034,Y,11.499993,0.465616
212687,1237662194004656453,A2034,Y,11.427743,0.421931
213329,1237664852576698371,A2065,Y,11.735382,0.056894
213377,1237662713679970440,A2065,Y,11.227840,0.055894
213275,1237665103827959858,A2065,Y,11.067132,0.741767
212936,1237671939804626951,A2255,Y,11.815869,0.350210
213043,1237651311069888680,A2255,Y,11.242548,0.895246
213031,1237651297648247146,A2255,Y,11.216073,0.247840


### Missing massive galaxies after the catalog update

**Purpose**: make sure that none of the most massive members of the previous catalog was lost in the update, and that every removal is understood.

**Procedure**
1. For each cluster, take the most massive members of the *previous* catalog.
2. Cross-match them with the *updated* catalog and list the ones that are missing.
3. Inspect the missing objects (SDSS image list, coordinates) to find the reason.

**Outcome**: the "missing" galaxies are not lost. They received a new objid when duplicate photometry was cleaned, so they are still in the updated catalog under another objid (listed in the next cell).

In [11]:
# Print the full rows for the "new" objids that correspond to each missing "old" objid
print(df[df['objid']==1237665442595209430]) # replacement for old objid: 1237665442595209429
print(df[df['objid']==1237662306194423877]) # replacement for old objid: 1237662306194423863
print(df[df['objid']==1237662194004656248]) # replacement for old objid: 1237662194004656247
print(df[df['objid']==1237664852576698387]) # replacement for old objid: 1237664852576698371
print(df[df['objid']==1237664852576633010]) # replacement for old objid: 1237662713679970440
print(df[df['objid']==1237671939804626959]) # replacement for old objid: 1237671939804626951
print(df[df['objid']==1237671939804626959]) # replacement for old objid: 1237671939804626951

                     objid          ra        dec  fiberMag_u  fiberMagerr_u  \
91757  1237665442595209430  209.642979  28.119242     19.7037       0.029587   

       fiberMag_g  fiberMagerr_g  fiberMag_r  fiberMagerr_r  fiberMag_i  \
91757    17.69488       0.004263    16.76546       0.003108    16.33248   

       fiberMagerr_i  fiberMag_z  fiberMagerr_z  petroMag_u  petroMagerr_u  \
91757       0.003478    15.99023       0.005099    17.89779       0.057924   

       petroMag_g  petroMagerr_g  petroMag_r  petroMagerr_r  petroMag_i  \
91757    15.92112        0.01569    15.00374       0.014428    14.55941   

       petroMagerr_i  petroMag_z  petroMagerr_z  modelMag_u  modelMagerr_u  \
91757       0.015244    14.28679       0.016423    17.86951       0.023488   

       modelMag_g  modelMagerr_g  modelMag_r  modelMagerr_r  modelMag_i  \
91757     15.8385       0.003096    14.92933        0.00246    14.49689   

       modelMagerr_i  modelMag_z  modelMagerr_z  cmodelMag_u  cmodelMage